In [ ]:
install.packages("rpart")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
library(rpart)

# Introducción

Casi todos los modelos que hemos utilizado para resolver problemas de aprendizaje supervisado (clasificación o regresión) se ven muy afectados ante la **variabilidad** de los datos.

Por ejemplo, si dividimos aleatoriamente los datos en dos subconjuntos y con cada uno de ellos entrenamos un árbol de clasificación, estos árboles pueden producir resultados muy diferentes.

Para construir un modelo con menos varianza podemos explorar varias técnicas. Una de ellas consiste en **construir varios modelos simples y combinarlos**, por ejemplo evaluando el **promedio** (en regresión) o a través de una **votación** (clasificación).

Algunas estrategias basadas en esta idea se exploran en las siguientes celdas.



# Bagging

Para aplicar la técnica de *bagging*, <font color="red">dividimos el conjunto de datos en $m$ subconjuntos</font> y utilizamos cada uno de esos subconjuntos para construir un **clasificador (o regresor) simple**.

Simbolicemos con $f_i$ al modelo construido con el subconjunto $i-$ésimo, entonces el modelo $f$ se puede obtener con el ensamble:

$$f(x_i) = \text{moda}\{f_1(x_i),\; f_2(x_i),\; f_3(x_i),\; \ldots,\; f_m(x_i)\}$$



### Ejemplo de aplicación

In [ ]:
dat <- read.csv("hr.csv")
head(dat)

,S,LPE,NP,ANH,TIC,Newborn,left
,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>
1,0.38,0.53,2,157,3,0,1
2,0.80,0.86,5,262,6,0,1
3,0.11,0.88,7,272,4,0,1
4,0.72,0.87,5,223,5,0,1
5,0.37,0.52,2,159,3,0,1
6,0.41,0.50,2,153,3,0,1


Dividimos en conjuntos de entrenamiento y de prueba

In [ ]:
set.seed(123)

n <- nrow(dat)
ind <- sample(1:n, 0.7*n)
train <- dat[ind,]
test <- dat[-ind,]

Separamos los datos de entrenamiento en tres subconjuntos

In [ ]:
# separamos aleatoriamente los datos en tres subconjuntos

n_train <- nrow(train)
i <- sample(1:n_train, n_train) #reordenamos aleatoriamente los índices
n_s <- n_train/3

i1 <- i[1:n_s]
i2 <- i[(n_s+1):(2*n_s)]
i3 <- i[(2*n_s+1):n_train]

train1 <- train[i1,]
train2 <- train[i2,]
train3 <- train[i3,]

Construimos tres árboles de clasificación

In [ ]:
t1 <- rpart(left ~ ., method="class", data=train1)
t2 <- rpart(left ~ ., method="class", data=train2)
t3 <- rpart(left ~ ., method="class", data=train3)

Probamos cada árbol sobre el conjunto de prueba


In [ ]:
yp1 <- predict(t1,test,type="class")
yp2 <- predict(t2,test,type="class")
yp3 <- predict(t3,test,type="class")

test$t1 <- as.integer(yp1)-1
test$t2 <- as.integer(yp2)-1
test$t3 <- as.integer(yp3)-1

head(test)
tail(test)

,S,LPE,NP,ANH,TIC,Newborn,left,t1,t2,t3
,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>
3,0.11,0.88,7,272,4,0,1,1,1,1
4,0.72,0.87,5,223,5,0,1,1,1,1
8,0.92,0.85,5,259,5,0,1,1,1,1
10,0.42,0.53,2,142,3,0,1,1,1,1
19,0.45,0.51,2,160,3,1,1,1,1,1
22,0.38,0.55,2,147,3,0,1,1,1,1


,S,LPE,NP,ANH,TIC,Newborn,left,t1,t2,t3
,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>
11967,0.92,0.65,3,176,3,0,0,0,0,0
11969,0.80,0.80,4,146,3,1,0,0,0,0
11974,0.49,0.71,4,178,5,0,0,0,0,0
11984,0.67,0.49,5,241,2,0,0,0,0,0
11988,0.19,0.79,4,229,4,0,0,0,0,0
11995,0.22,0.94,3,193,6,0,0,0,0,0


Asignamos la clase tomando en cuenta a la moda entre las predicciones de los 3 árboles.

In [ ]:
#bagging
test$pred <- test$t1+test$t2+test$t3
test$pred <- ifelse(test$pred/3>=0.5,1,0)
head(test)
tail(test)

,S,LPE,NP,ANH,TIC,Newborn,left,t1,t2,t3,pred
,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
3,0.11,0.88,7,272,4,0,1,1,1,1,1
4,0.72,0.87,5,223,5,0,1,1,1,1,1
8,0.92,0.85,5,259,5,0,1,1,1,1,1
10,0.42,0.53,2,142,3,0,1,1,1,1,1
19,0.45,0.51,2,160,3,1,1,1,1,1,1
22,0.38,0.55,2,147,3,0,1,1,1,1,1


,S,LPE,NP,ANH,TIC,Newborn,left,t1,t2,t3,pred
,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
11967,0.92,0.65,3,176,3,0,0,0,0,0,0
11969,0.80,0.80,4,146,3,1,0,0,0,0,0
11974,0.49,0.71,4,178,5,0,0,0,0,0,0
11984,0.67,0.49,5,241,2,0,0,0,0,0,0
11988,0.19,0.79,4,229,4,0,0,0,0,0,0
11995,0.22,0.94,3,193,6,0,0,0,0,0,0


Matriz de confusión

In [ ]:
table(test$left, test$pred)

   
       0    1
  0 3002   26
  1   45  527

In [ ]:
# exactitud
(527+3002)/(527+3002+45+26)

[1] 0.9802778

In [ ]:
#exactitud de referencia
1-mean(test$left)

[1] 0.8411111

### Bootstrapping

La dificultad práctica para la aplicación del bagging es la necesidad de contar con un conjunto de datos suficientemente grande para que pueda subdividirse en $M$ subconjuntos. Esta dificultad puede superarse mediante la aplicación del **bootstrapping**. Boostrapping, es una técnica mediante la cual se muestrea un conjunto, con reemplazo, para la construcción de varios subconjuntos.  


In [ ]:
#en la implementación de bagging obtuvimos tres subconjuntos con 2800 observaciones
length(i1)
length(i2)
length(i3)

[1] 2800

[1] 2800

[1] 2800

**Remuestreamos** ahora 3 veces el conjunto de entrenamiento. Cada subconjunto contiene el mismo número de elementos que el original. Observa que utilizamos un **muestreo con reemplazo**.

In [ ]:
# separamos aleatoriamente los datos en tres subconjuntos
i1 <- sample(1:n_train, n_train, replace = T) #reordenamos aleatoriamente los índices
i2 <- sample(1:n_train, n_train, replace = T) #reordenamos aleatoriamente los índices
i3 <- sample(1:n_train, n_train, replace = T) #reordenamos aleatoriamente los índices

train1 <- train[i1,]
train2 <- train[i2,]
train3 <- train[i3,]

In [ ]:
nrow(train1)
nrow(train2)
nrow(train3)

[1] 8400

[1] 8400

[1] 8400

In [ ]:
t1 <- rpart(left ~ ., method="class", data=train1)
t2 <- rpart(left ~ ., method="class", data=train2)
t3 <- rpart(left ~ ., method="class", data=train3)

In [ ]:
yp1 <- predict(t1,test,type="class")
yp2 <- predict(t2,test,type="class")
yp3 <- predict(t3,test,type="class")

test$t1 <- as.integer(yp1)-1
test$t2 <- as.integer(yp2)-1
test$t3 <- as.integer(yp3)-1

head(test)
tail(test)

,S,LPE,NP,ANH,TIC,Newborn,left,t1,t2,t3,pred
,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
3,0.11,0.88,7,272,4,0,1,1,1,1,1
4,0.72,0.87,5,223,5,0,1,1,1,1,1
8,0.92,0.85,5,259,5,0,1,1,1,1,1
10,0.42,0.53,2,142,3,0,1,1,1,1,1
19,0.45,0.51,2,160,3,1,1,1,1,1,1
22,0.38,0.55,2,147,3,0,1,1,1,1,1


,S,LPE,NP,ANH,TIC,Newborn,left,t1,t2,t3,pred
,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
11967,0.92,0.65,3,176,3,0,0,0,0,0,0
11969,0.80,0.80,4,146,3,1,0,0,0,0,0
11974,0.49,0.71,4,178,5,0,0,0,0,0,0
11984,0.67,0.49,5,241,2,0,0,0,0,0,0
11988,0.19,0.79,4,229,4,0,0,0,0,0,0
11995,0.22,0.94,3,193,6,0,0,0,0,0,0


In [ ]:
#bagging

test$pred <- test$t1+test$t2+test$t3
test$pred <- ifelse(test$pred/3>=0.5,1,0)
head(test)
tail(test)

,S,LPE,NP,ANH,TIC,Newborn,left,t1,t2,t3,pred
,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
3,0.11,0.88,7,272,4,0,1,1,1,1,1
4,0.72,0.87,5,223,5,0,1,1,1,1,1
8,0.92,0.85,5,259,5,0,1,1,1,1,1
10,0.42,0.53,2,142,3,0,1,1,1,1,1
19,0.45,0.51,2,160,3,1,1,1,1,1,1
22,0.38,0.55,2,147,3,0,1,1,1,1,1


,S,LPE,NP,ANH,TIC,Newborn,left,t1,t2,t3,pred
,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
11967,0.92,0.65,3,176,3,0,0,0,0,0,0
11969,0.80,0.80,4,146,3,1,0,0,0,0,0
11974,0.49,0.71,4,178,5,0,0,0,0,0,0
11984,0.67,0.49,5,241,2,0,0,0,0,0,0
11988,0.19,0.79,4,229,4,0,0,0,0,0,0
11995,0.22,0.94,3,193,6,0,0,0,0,0,0


In [ ]:
table(test$left, test$pred)

   
       0    1
  0 3007   21
  1   46  526

In [ ]:
# exactitud
(3007+526)/(3007+526+21+46)

[1] 0.9813889

# Otros métodos de ensamble

## Random Forest

Cuando se utilizan los áboles como modelo básico en la técnica de bagging, se presenta el inconveniente de que la mayoría de los árboles construidos se correlacionan fuertemente (su estructura puede ser similar debido a que se basa en unos cuantos **predictores dominantes**).

Para superar esta limitación, se puede restringir el número de predictores que serán explorados en cada división durante la construcción de cada árbol. Por ejemplo, si se dispone de $p$ predictores, pueden seleccionarse aleatoriamente $m$ de ellos: $$m\approx \sqrt{p}$$

## Boosting

El boosting es una técnica similar al bagging que permite construir modelos **de forma secuencial**: cada modelo se construye a partir de la información de los modelos previos.

**El algoritmo se aplica de forma general en problemas de regresión**, aunque puede adaptarse para clasificación.

La idea es generar un primer modelo que se ajuste a los datos. Los posteriores modelos deberán enfocarse en explicar los residuos del primer modelo.

El algoritmo se describe en la siguiente imagen:


<img src = "https://drive.google.com/uc?export=view&id=1zGof9WfZ7vGNMOUODn1sif9FrqvB14zk" width="500"/>



## Adaboost

El algoritmo de **adaboost** es uno de los métodos de ensamble más aplicados en problemas de clasificación binarios. El algoritmo se describe en la siguiente imagen:

<img src = "https://drive.google.com/uc?export=view&id=1xvc0WETfInuhe5CbI14OeXtQfJmaP8jH" width="700"/>





# Implementación en R

Puedes encontrar detalles de las implementaciones de estos algoritmos desde R en las siguienets páginas:


https://www.rdocumentation.org/packages/ipred/versions/0.4-0/topics/bagging
https://www.rdocumentation.org/packages/JOUSBoost/versions/2.1.0/topics/adaboost
https://www.rdocumentation.org/packages/randomForest/versions/4.6-14/topics/randomForest


In [ ]:
install.packages("randomForest")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
library(randomForest)
library(dplyr)

randomForest 4.7-1.2

Type rfNews() to see new features/changes/bug fixes.


Attaching package: ‘dplyr’


The following object is masked from ‘package:randomForest’:

    combine


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
train$left_f <- as.factor(train$left)
summary(train$left_f)

0    1 
6972 1428

In [ ]:
head(train)

,S,LPE,NP,ANH,TIC,Newborn,left,left_f
,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<fct>
2463,0.63,0.80,3,205,2,0,0,0
2511,0.76,0.50,3,258,3,0,0,0
10419,0.62,0.49,4,174,3,0,0,0
8718,0.45,0.41,3,193,2,1,0,0
2986,0.72,0.63,4,241,4,1,0,0
1842,0.84,1.00,5,242,5,0,1,1


In [ ]:
rf <- randomForest(left_f~S+LPE+NP+ANH+TIC+Newborn, ntree=100, data=train)

In [ ]:
yp <- predict(rf, test)
t <- table(test$left, yp)
t

   yp
       0    1
  0 3025    3
  1   43  529

In [ ]:
(t[1,1]+t[2,2])/(sum(t))

[1] 0.9872222